In [61]:
%load_ext autoreload
%autoreload 2

from scalargrad.nn import MLP
from scalargrad.value import Value
n = MLP(3, [4, 4, 1], output_activation=Value.tanh)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [62]:
len(n.parameters())

41

In [63]:
xs: list[list[Value | float]] = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0], 
]

ys = [[1.0], [-1.0], [-1.0], [1.0]]
ypred = [n(x) for x in xs]
ypred

[[Value(data=0.5639262127781015, grad=0.0) , op=tanh],
 [Value(data=0.6111087824545287, grad=0.0) , op=tanh],
 [Value(data=0.5790863112258993, grad=0.0) , op=tanh],
 [Value(data=0.38129361142296153, grad=0.0) , op=tanh]]

In [64]:
target, max_steps, lr = 0.001, 2000, 0.01

loss = Value(float("inf"))
for k in range(1, max_steps + 1):
    ypred = [n(x)[0] for x in xs]
    loss = sum(((yp - y[0])**2 for y, yp in zip(ys, ypred)), Value(0.0))

    if k == 1 or k % 100 == 0:
        print(f"step {k}: loss {loss.data:.6f}")
    if loss.data <= target:
        print(f"Reached target loss {target} at step {k}: loss = {loss.data:.6f}")
        break

    n.zero_grad()
    loss.backward()
    for p in n.parameters():
        p.data -= lr * p.grad
else:
    print(f"Hit the step limit ({max_steps}): loss = {loss.data:.6f}")

step 1: loss 5.662143
step 100: loss 0.089570
step 200: loss 0.021166
step 300: loss 0.010468
step 400: loss 0.006604
step 500: loss 0.004698
step 600: loss 0.003590
step 700: loss 0.002875
step 800: loss 0.002381
step 900: loss 0.002021
step 1000: loss 0.001748
step 1100: loss 0.001536
step 1200: loss 0.001366
step 1300: loss 0.001227
step 1400: loss 0.001112
step 1500: loss 0.001015
Reached target loss 0.001 at step 1518: loss = 0.000999


In [65]:
ypred = [n(x) for x in xs]
ypred

[[Value(data=0.9965549999767003, grad=0.0) , op=tanh],
 [Value(data=-0.999739290581321, grad=0.0) , op=tanh],
 [Value(data=-0.9834945742511642, grad=0.0) , op=tanh],
 [Value(data=0.9732617686597727, grad=0.0) , op=tanh]]

In [66]:
acts = []
for x in xs:
    h = x
    per_layer = []
    for layer in n.layers:
        h = layer(h)
        per_layer.append([v.data for v in h])
    acts.append(per_layer)

for li in range(len(n.layers) - 1):
    for ni in range(len(n.layers[li].neurons)):
        values = [acts[s][li][ni] for s in range(len(xs))]
        dead = all(v == 0 for v in values)
        print(f"layer {li} neuron {ni}: {[round(v, 3) for v in values]}"
              f"{'  ← DEAD' if dead else ''}")

layer 0 neuron 0: [0, 0, 0, 0]  ← DEAD
layer 0 neuron 1: [0.351, 2.368, 1.332, 0]
layer 0 neuron 2: [3.67, 0.554, 1.398, 1.566]
layer 0 neuron 3: [0.13, 2.309, 1.626, 0.0]
layer 1 neuron 0: [2.363, 0, 0.34, 1.298]
layer 1 neuron 1: [0.307, 4.192, 2.643, 0]
layer 1 neuron 2: [1.84, 0, 0, 0.917]
layer 1 neuron 3: [0.435, 1.48, 1.077, 0.18]
